In [1]:
"""
Scrape 2010 headlines from multiple news sites using the Wayback Machine (archive.org).

Why Wayback Machine instead of scraping the live sites?
    - Live BBC/Bloomberg/etc. pages only show CURRENT news, not 2010 news.
    - Bloomberg is paywalled + has strong anti-bot protection; direct scraping fails.
    - archive.org has snapshots of most site homepages from 2010, and works the
      same way for every site, so one script covers all of them.

How it works:
    1. For each site + each month of 2010, ask the Wayback CDX API for the closest
       saved snapshot of that site's homepage.
    2. Fetch that archived homepage HTML.
    3. Pull out headline text from <a> tags (best-effort, works reasonably well on
       most news homepages since headlines are almost always links).
    4. Write everything to a CSV: site, snapshot_date, headline, link.

Setup:
    pip install requests beautifulsoup4

Run:
    python scrape_2010_headlines.py

Notes:
    - This is best-effort. 2010-era HTML varies a lot between sites, so headline
      extraction quality will vary. For a cleaner result on a specific site, you'd
      want a custom CSS selector for that site's markup (there's a hook for that
      below in SITE_SELECTORS).
    - Be polite: script sleeps between requests. archive.org can be slow/rate-limit
      under heavy use.
"""

import csv
import time
import requests
from bs4 import BeautifulSoup

# ---- Edit this list with your actual target sites ----
SITES = [
    "bbc.com",
    "bloomberg.com",
    "cnn.com",
    "reuters.com",
    "nytimes.com",
    "theguardian.com",
    "washingtonpost.com",
    "wsj.com",
    "aljazeera.com",
    "foxnews.com",
]

# Optional: custom CSS     selector per site for better headline extraction.
# If a site isn't listed here, the script falls back to a generic <a> scan.
SITE_SELECTORS = {
    # "bbc.com": "a.gs-c-promo-heading",
}

YEAR = 2010
MONTHS = range(1, 13)
OUTPUT_FILE = "headlines_2010.csv"
SECONDS_BETWEEN_REQUESTS = 2
MIN_HEADLINE_LENGTH = 15  # filters out nav links like "Home", "Sign in"

CDX_API = "https://web.archive.org/cdx/search/cdx"
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: you@example.com)"}


def get_snapshot_url(site: str, year: int, month: int) -> str | None:
    """Find the closest archived snapshot of a site's homepage for a given month."""
    timestamp = f"{year}{month:02d}15"  # aim for the middle of the month
    params = {
        "url": site,
        "timestamp": timestamp,
        "output": "json",
        "limit": 1,
        "filter": "statuscode:200",
    }
    try:
        resp = requests.get(CDX_API, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        data = resp.json()
    except (requests.RequestException, ValueError):
        return None

    if len(data) < 2:  # first row is header
        return None

    row = data[1]
    snapshot_ts = row[1]
    original_url = row[2]
    return f"https://web.archive.org/web/{snapshot_ts}/{original_url}"


def extract_headlines(html: str, site: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    selector = SITE_SELECTORS.get(site)
    if selector:
        elements = soup.select(selector)
    else:
        elements = soup.find_all("a")

    headlines = []
    seen = set()
    for el in elements:
        text = el.get_text(strip=True)
        if len(text) >= MIN_HEADLINE_LENGTH and text not in seen:
            seen.add(text)
            headlines.append(text)

    return headlines


def scrape_site_month(site: str, year: int, month: int) -> list[dict]:
    snapshot_url = get_snapshot_url(site, year, month)
    if not snapshot_url:
        print(f"  {site} {year}-{month:02d}: no snapshot found")
        return []

    try:
        resp = requests.get(snapshot_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  {site} {year}-{month:02d}: fetch failed ({e})")
        return []

    headlines = extract_headlines(resp.text, site)
    print(f"  {site} {year}-{month:02d}: {len(headlines)} headlines from {snapshot_url}")

    return [
        {
            "site": site,
            "snapshot_date": f"{year}-{month:02d}",
            "snapshot_url": snapshot_url,
            "headline": h,
        }
        for h in headlines
    ]


def main():
    fieldnames = ["site", "snapshot_date", "snapshot_url", "headline"]

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        total = 0
        for site in SITES:
            print(f"\nScraping {site}...")
            for month in MONTHS:
                rows = scrape_site_month(site, YEAR, month)
                for row in rows:
                    writer.writerow(row)
                total += len(rows)
                time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"\nDone. Saved {total} headlines to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Scraping bbc.com...
  bbc.com 2010-01: no snapshot found
  bbc.com 2010-02: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-03: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-04: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-05: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-06: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-07: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-08: no snapshot found
  bbc.com 2010-09: no snapshot found
  bbc.com 2010-10: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/
  bbc.com 2010-11: no snapshot found
  bbc.com 2010-12: 0 headlines from https://web.archive.org/web/20000301003115/http://www.bbc.com:80/

Scraping bloom

In [2]:
"""
Scrape 2010 headlines from multiple news sites using the Wayback Machine (archive.org).

Why Wayback Machine instead of scraping the live sites?
    - Live BBC/Bloomberg/etc. pages only show CURRENT news, not 2010 news.
    - Bloomberg is paywalled + has strong anti-bot protection; direct scraping fails.
    - archive.org has snapshots of most site homepages from 2010, and works the
      same way for every site, so one script covers all of them.

How it works:
    1. For each site + each month of 2010, ask the Wayback CDX API for the closest
       saved snapshot of that site's homepage.
    2. Fetch that archived homepage HTML.
    3. Pull out headline text from <a> tags (best-effort, works reasonably well on
       most news homepages since headlines are almost always links).
    4. Write everything to a CSV: site, snapshot_date, headline, link.

Setup:
    pip install requests beautifulsoup4

Run:
    python scrape_2010_headlines.py

Notes:
    - This is best-effort. 2010-era HTML varies a lot between sites, so headline
      extraction quality will vary. For a cleaner result on a specific site, you'd
      want a custom CSS selector for that site's markup (there's a hook for that
      below in SITE_SELECTORS).
    - Be polite: script sleeps between requests. archive.org can be slow/rate-limit
      under heavy use.
"""

import csv
import time
import requests
from bs4 import BeautifulSoup

# ---- Edit this list with your actual target sites ----
SITES = [
    "bbc.com",
    "bloomberg.com",
    "cnn.com",
    "reuters.com",
    "nytimes.com",
    "theguardian.com",
    "washingtonpost.com",
    "wsj.com",
    "aljazeera.com",
    "foxnews.com",
]

# Optional: custom CSS selector per site for better headline extraction.
# If a site isn't listed here, the script falls back to a generic <a> scan.
SITE_SELECTORS = {
    # "bbc.com": "a.gs-c-promo-heading",
}

YEAR = 2010
MONTHS = range(1, 13)
OUTPUT_FILE = "headlines_2010.csv"
SECONDS_BETWEEN_REQUESTS = 2
MIN_HEADLINE_LENGTH = 15  # filters out nav links like "Home", "Sign in"

AVAILABILITY_API = "https://archive.org/wayback/available"
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: you@example.com)"}


def get_snapshot_url(site: str, year: int, month: int) -> str | None:
    """Find the closest archived snapshot of a site's homepage for a given month
    using the Wayback Availability API (returns the nearest snapshot to the
    requested timestamp, unlike the raw CDX API which can return the oldest
    match instead of the closest one)."""
    timestamp = f"{year}{month:02d}15"  # aim for the middle of the month
    params = {"url": site, "timestamp": timestamp}

    try:
        resp = requests.get(AVAILABILITY_API, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        data = resp.json()
    except (requests.RequestException, ValueError):
        return None

    snapshot = data.get("archived_snapshots", {}).get("closest")
    if not snapshot or not snapshot.get("available"):
        return None

    snapshot_url = snapshot["url"]
    snapshot_ts = snapshot.get("timestamp", "")

    # Sanity check: skip snapshots wildly outside the requested year
    # (the availability API can occasionally return a distant fallback
    # if nothing near the target date exists).
    if snapshot_ts[:4] != str(year):
        return None

    return snapshot_url


def extract_headlines(html: str, site: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    selector = SITE_SELECTORS.get(site)
    if selector:
        elements = soup.select(selector)
    else:
        elements = soup.find_all("a")

    headlines = []
    seen = set()
    for el in elements:
        text = el.get_text(strip=True)
        if len(text) >= MIN_HEADLINE_LENGTH and text not in seen:
            seen.add(text)
            headlines.append(text)

    return headlines


def scrape_site_month(site: str, year: int, month: int) -> list[dict]:
    snapshot_url = get_snapshot_url(site, year, month)
    if not snapshot_url:
        print(f"  {site} {year}-{month:02d}: no snapshot found")
        return []

    try:
        resp = requests.get(snapshot_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  {site} {year}-{month:02d}: fetch failed ({e})")
        return []

    headlines = extract_headlines(resp.text, site)
    print(f"  {site} {year}-{month:02d}: {len(headlines)} headlines from {snapshot_url}")

    return [
        {
            "site": site,
            "snapshot_date": f"{year}-{month:02d}",
            "snapshot_url": snapshot_url,
            "headline": h,
        }
        for h in headlines
    ]


def main():
    fieldnames = ["site", "snapshot_date", "snapshot_url", "headline"]

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        total = 0
        for site in SITES:
            print(f"\nScraping {site}...")
            for month in MONTHS:
                rows = scrape_site_month(site, YEAR, month)
                for row in rows:
                    writer.writerow(row)
                total += len(rows)
                time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"\nDone. Saved {total} headlines to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Scraping bbc.com...
  bbc.com 2010-01: no snapshot found
  bbc.com 2010-02: no snapshot found
  bbc.com 2010-03: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-04: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-05: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-06: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-07: no snapshot found
  bbc.com 2010-08: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-09: 104 headlines from http://web.archive.org/web/20101015031310/http://www.bbc.com/
  bbc.com 2010-10: 104 headlines from http://web.archive.org/web/20101015125551/http://www.bbc.com/
  bbc.com 2010-11: no snapshot found
  bbc.com 2010-12: fetch failed (503 Server Error: Service Unavailable for url: http://web.archive.org/web/20101215235257/http://ww

In [ ]:
"""
Scrape headlines from multiple news sites, from 2010 to present, using the
Wayback Machine (archive.org).

This is the same approach as the 2010-only version, extended to loop over
every year from 2010 to the current year. Given the much larger volume of
requests (years x sites x 12 months), this version adds:

    - Resumable progress: already-scraped site/month combos are tracked in a
      progress file, so if the script crashes or you stop it, re-running picks
      up where it left off instead of starting over.
    - Incremental CSV writing: each row is written (and flushed) as soon as
      it's found, so you don't lose everything if it's interrupted.
    - Retry with backoff: if archive.org rate-limits or times out, the script
      waits and retries instead of just failing that month.

Setup:
    pip install requests beautifulsoup4

Run:
    python scrape_2010_to_present.py

To resume after an interruption, just run it again — it skips anything
already recorded in PROGRESS_FILE.

To start completely fresh, delete both OUTPUT_FILE and PROGRESS_FILE.

Notes:
    - Volume estimate: ~17 years x 10 sites x 12 months ~= 2,000 snapshot
      lookups. At ~2-3 seconds between requests, expect this to take
      roughly 1-2 hours. Run it, let it work in the background, and resume
      if needed.
    - Headline extraction is best-effort (see SITE_SELECTORS to improve
      accuracy for a specific site's markup).
"""

import csv
import json
import os
import time
from datetime import datetime

import requests
from bs4 import BeautifulSoup

# ---- Edit this list with your actual target sites ----
SITES = [
    "bbc.com",
    "bloomberg.com", 
    "cnn.com",
    "reuters.com",
    "nytimes.com",
    "theguardian.com",
    "washingtonpost.com",
    "wsj.com",
    "aljazeera.com",
    "foxnews.com",
]

# Optional: custom CSS selector per site for better headline extraction.
# If a site isn't listed here, the script falls back to a generic <a> scan.
SITE_SELECTORS = {
    # "bbc.com": "a.gs-c-promo-heading",
}

START_YEAR = 2010
END_YEAR = datetime.now().year
MONTHS = range(1, 13)

OUTPUT_FILE = "headlines_2010_to_present.csv"
PROGRESS_FILE = "headlines_progress.json"

SECONDS_BETWEEN_REQUESTS = 2
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 10
MIN_HEADLINE_LENGTH = 15  # filters out nav links like "Home", "Sign in"

AVAILABILITY_API = "https://archive.org/wayback/available"
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: you@example.com)"}
FIELDNAMES = ["site", "snapshot_date", "snapshot_url", "headline"]


def load_progress() -> set:
    if not os.path.exists(PROGRESS_FILE):
        return set()
    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        return set(json.load(f))


def save_progress(done: set):
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(done), f)


def request_with_retry(url: str, params: dict = None) -> requests.Response | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if resp.status_code == 429:
                wait = RETRY_BACKOFF_SECONDS * attempt
                print(f"    rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                print(f"    failed after {MAX_RETRIES} attempts: {e}")
                return None
            wait = RETRY_BACKOFF_SECONDS * attempt
            print(f"    error ({e}), retrying in {wait}s...")
            time.sleep(wait)
    return None


def get_snapshot_url(site: str, year: int, month: int) -> str | None:
    """Find the closest archived snapshot of a site's homepage for a given month
    using the Wayback Availability API (returns the nearest snapshot to the
    requested date, unlike the raw CDX API which can return the oldest match
    on record instead of the closest one)."""
    timestamp = f"{year}{month:02d}15"  # aim for the middle of the month
    resp = request_with_retry(AVAILABILITY_API, params={"url": site, "timestamp": timestamp})
    if resp is None:
        return None

    try:
        data = resp.json()
    except ValueError:
        return None

    snapshot = data.get("archived_snapshots", {}).get("closest")
    if not snapshot or not snapshot.get("available"):
        return None

    snapshot_ts = snapshot.get("timestamp", "")
    # Sanity check: skip snapshots wildly outside the requested year
    if snapshot_ts[:4] != str(year):
        return None

    return snapshot["url"]


def extract_headlines(html: str, site: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    selector = SITE_SELECTORS.get(site)
    elements = soup.select(selector) if selector else soup.find_all("a")

    headlines, seen = [], set()
    for el in elements:
        text = el.get_text(strip=True)
        if len(text) >= MIN_HEADLINE_LENGTH and text not in seen:
            seen.add(text)
            headlines.append(text)

    return headlines


def scrape_site_month(site: str, year: int, month: int) -> list[dict]:
    snapshot_url = get_snapshot_url(site, year, month)
    if not snapshot_url:
        print(f"  {site} {year}-{month:02d}: no snapshot found")
        return []

    resp = request_with_retry(snapshot_url)
    if resp is None:
        return []

    headlines = extract_headlines(resp.text, site)
    print(f"  {site} {year}-{month:02d}: {len(headlines)} headlines")

    return [
        {
            "site": site,
            "snapshot_date": f"{year}-{month:02d}",
            "snapshot_url": snapshot_url,
            "headline": h,
        }
        for h in headlines
    ]


def main():
    done = load_progress()
    file_exists = os.path.exists(OUTPUT_FILE)

    total_new = 0
    with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()

        for site in SITES:
            print(f"\nScraping {site}...")
            for year in range(START_YEAR, END_YEAR + 1):
                for month in MONTHS:
                    if year == END_YEAR and month > datetime.now().month:
                        continue  # don't try to fetch future months

                    key = f"{site}|{year}-{month:02d}"
                    if key in done:
                        continue

                    rows = scrape_site_month(site, year, month)
                    for row in rows:
                        writer.writerow(row)
                    f.flush()

                    total_new += len(rows)
                    done.add(key)
                    save_progress(done)

                    time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"\nDone. Added {total_new} new headlines to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Scraping bbc.com...
